# Notebook 3: Entrenamiento y Comparativa de Arquitecturas LSP

Modelos entrenados:
1. **CNN-LSTM** (ResNet50 + BiLSTM) — Baseline
2. **ST-GCN** (Spatial-Temporal GCN sobre landmarks)
3. **Fusión Multimodal** (CNN-LSTM + ST-GCN)

Métricas: Accuracy, F1-macro, Latencia de inferencia, Matriz de confusión

In [1]:
# Dependencias ya instaladas en .venv310
# !pip install -q torch torchvision transformers scikit-learn wandb
# !pip install -q mediapipe


In [2]:
import os, sys, json, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path

# Configurar paths
# Detectar entorno Colab vs Local
try:
    from google.colab import drive; IN_COLAB = True
    sys.path.insert(0, '/content/drive/MyDrive/TRADUCTOR_LSP')
except ImportError:
    IN_COLAB = False
    from pathlib import Path
    _nb_path = Path.cwd()
    _base = str(_nb_path.parent if _nb_path.name == 'notebooks' else _nb_path)
    sys.path.insert(0, _base)
    import os; os.chdir(_base)

from src.models import CNNLSTM, STGCN, LSPFusionModel
from src.dataset.lsp_dataset import get_dataloaders
from src.training.trainer import LSPTrainer
from src.training.metrics import (
    compute_metrics, plot_confusion_matrix, plot_training_curves
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cpu


In [3]:
# ── Configuración global ──────────────────────────────────────────────
MANIFEST    = 'data/manifest_segments.csv'
PIXELS_DIR  = 'data/processed_pixels'
KP_DIR      = 'data/landmarks'
CKPT_DIR    = 'checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

with open('data/label2idx.json') as f:
    label2idx = json.load(f)
with open('data/class_weights.json') as f:
    class_weights_dict = json.load(f)

N_CLASSES = len(label2idx)
print(f'Clases LSP: {N_CLASSES}')
print(f'Clases: {list(label2idx.keys())}')

# class_weights_dict tiene claves tipo 'vineta_002' → convertir a índice entero
# con label2idx['vineta_002'] = 0, etc.
class_weights_by_idx = {label2idx[cls]: w for cls, w in class_weights_dict.items()
                        if cls in label2idx}

class_weights_tensor = torch.tensor(
    [class_weights_by_idx.get(i, 1.0) for i in range(N_CLASSES)],
    dtype=torch.float32,
)
print(f'class_weights_tensor: {class_weights_tensor.tolist()[:5]} ...')

Clases LSP: 26
Clases: ['vineta_002', 'vineta_003', 'vineta_004', 'vineta_005', 'vineta_006', 'vineta_008', 'vineta_011', 'vineta_012', 'vineta_013', 'vineta_014', 'vineta_015', 'vineta_017', 'vineta_018', 'vineta_019', 'vineta_020', 'vineta_021', 'vineta_022', 'vineta_023', 'vineta_024', 'vineta_025', 'vineta_027', 'vineta_029', 'vineta_030', 'vineta_037', 'vineta_042', 'vineta_043']
class_weights_tensor: [1.384424090385437, 0.24473987519741058, 1.1891847848892212, 1.0743985176086426, 0.7171887159347534] ...


In [4]:
# ── DataLoaders ──────────────────────────────────────────────────────
# Para CNN-LSTM y VideoMAE (pixels)
loaders_pixels = get_dataloaders(
    manifest_csv=MANIFEST,
    label2idx=label2idx,
    mode='pixels',
    pixels_dir=PIXELS_DIR,
    batch_size=8,
    num_workers=2,
    class_weights=class_weights_by_idx,
)

# Para ST-GCN (landmarks)
loaders_kp = get_dataloaders(
    manifest_csv=MANIFEST,
    label2idx=label2idx,
    mode='landmarks',
    landmarks_dir=KP_DIR,
    batch_size=16,
    num_workers=2,
    class_weights=class_weights_by_idx,
)

# Para fusión multimodal
loaders_both = get_dataloaders(
    manifest_csv=MANIFEST,
    label2idx=label2idx,
    mode='both',
    pixels_dir=PIXELS_DIR,
    landmarks_dir=KP_DIR,
    batch_size=8,
    num_workers=2,
    class_weights=class_weights_by_idx,
)

print(f"Train batches (pixels): {len(loaders_pixels['train'])}")
print(f"Train batches (kp):     {len(loaders_kp['train'])}")
print(f"Train batches (both):   {len(loaders_both['train'])}")

Train batches (pixels): 631
Train batches (kp):     315
Train batches (both):   631


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# MODELO 1: CNN-LSTM (Baseline)
# ══════════════════════════════════════════════════════════════════════
print('=' * 60)
print('ENTRENANDO: CNN-LSTM (ResNet50 + BiLSTM)')
print('=' * 60)

cnn_lstm = CNNLSTM(
    n_classes=N_CLASSES,
    hidden_size=512,
    num_layers=2,
    dropout=0.3,
    pretrained=True,
    freeze_backbone_epochs=5,
)

n_params = sum(p.numel() for p in cnn_lstm.parameters() if p.requires_grad)
print(f'Parámetros entrenables: {n_params:,}')

trainer_cnn = LSPTrainer(
    model=cnn_lstm,
    loaders=loaders_pixels,
    mode='pixels',
    n_classes=N_CLASSES,
    device=DEVICE,
    output_dir=CKPT_DIR,
    lr=1e-4,
    weight_decay=1e-4,
    epochs=50,
    patience=10,
    class_weights=class_weights_tensor,
    mixed_precision=True,
    wandb_project='traductor-lsp',
    model_name='cnn_lstm',
)

history_cnn = trainer_cnn.train()
test_cnn = trainer_cnn.evaluate_test(f'{CKPT_DIR}/cnn_lstm_best.pt')

ENTRENANDO: CNN-LSTM (ResNet50 + BiLSTM)
Parámetros entrenables: 11,821,338

Iniciando entrenamiento — cnn_lstm
Device: cpu | Épocas: 50 | Modo: pixels



/Users/usuario/Documents/TRADUCTOR_LSP/.venv310/lib/python3.10/site-packages/torch/amp/autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# MODELO 2: ST-GCN sobre landmarks
# ══════════════════════════════════════════════════════════════════════
print('=' * 60)
print('ENTRENANDO: ST-GCN (Spatial-Temporal GCN sobre MediaPipe)')
print('=' * 60)

stgcn = STGCN(
    n_classes=N_CLASSES,
    n_nodes=75,          # 42 manos + 33 pose
    in_channels=3,
    hidden_channels=64,
    num_layers=4,
    dropout=0.3,
)

n_params_st = sum(p.numel() for p in stgcn.parameters() if p.requires_grad)
print(f'Parámetros entrenables: {n_params_st:,}')

trainer_st = LSPTrainer(
    model=stgcn,
    loaders=loaders_kp,
    mode='landmarks',
    n_classes=N_CLASSES,
    device=DEVICE,
    output_dir=CKPT_DIR,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=50,
    patience=10,
    class_weights=class_weights_tensor,
    mixed_precision=True,
    wandb_project='traductor-lsp',
    model_name='stgcn',
)

history_st = trainer_st.train()
test_st = trainer_st.evaluate_test(f'{CKPT_DIR}/stgcn_best.pt')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# MODELO 3: Fusión Multimodal CNN-LSTM + ST-GCN
# ══════════════════════════════════════════════════════════════════════
print('=' * 60)
print('ENTRENANDO: Fusión Multimodal (CNN-LSTM + ST-GCN)')
print('=' * 60)

# Reutilizar backbones ya entrenados como inicialización
pixel_backbone_pretrained = CNNLSTM(
    n_classes=N_CLASSES, hidden_size=512, pretrained=True
)
ckpt = torch.load(f'{CKPT_DIR}/cnn_lstm_best.pt', map_location='cpu')
pixel_backbone_pretrained.load_state_dict(ckpt['model_state'])

kp_backbone_pretrained = STGCN(n_classes=N_CLASSES, n_nodes=75)
ckpt_st = torch.load(f'{CKPT_DIR}/stgcn_best.pt', map_location='cpu')
kp_backbone_pretrained.load_state_dict(ckpt_st['model_state'])

fusion_model = LSPFusionModel(
    pixel_backbone=pixel_backbone_pretrained,
    landmark_backbone=kp_backbone_pretrained,
    dim_pixels=512 * 2,   # BiLSTM hidden*2
    dim_landmarks=64 * 4, # ST-GCN final channels
    n_classes=N_CLASSES,
    fusion_strategy='concat',
    hidden_dim=512,
    dropout=0.3,
)

trainer_fusion = LSPTrainer(
    model=fusion_model,
    loaders=loaders_both,
    mode='both',
    n_classes=N_CLASSES,
    device=DEVICE,
    output_dir=CKPT_DIR,
    lr=5e-5,
    weight_decay=1e-4,
    epochs=30,
    patience=8,
    class_weights=class_weights_tensor,
    mixed_precision=True,
    wandb_project='traductor-lsp',
    model_name='fusion',
)

history_fusion = trainer_fusion.train()
test_fusion = trainer_fusion.evaluate_test(f'{CKPT_DIR}/fusion_best.pt')

In [ ]:
# ── Comparativa final de modelos ──────────────────────────────────────
results = {
    'CNN-LSTM': test_cnn,
    'ST-GCN':   test_st,
    'Fusión':   test_fusion,
}

df_results = pd.DataFrame(results).T.round(4)
print('\n' + '=' * 60)
print('COMPARATIVA DE MODELOS — TEST SET')
print('=' * 60)
print(df_results.to_string())

# Gráfico comparativo
metrics_to_plot = ['accuracy', 'f1_macro', 'f1_weighted']
df_plot = df_results[metrics_to_plot]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_plot))
w = 0.25
colors = ['steelblue', 'coral', 'mediumseagreen']

for i, metric in enumerate(metrics_to_plot):
    bars = ax.bar(x + i*w, df_plot[metric], w, label=metric, color=colors[i], alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + w)
ax.set_xticklabels(df_plot.index, fontsize=11)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.15)
ax.set_title('Comparativa de Arquitecturas — LSP Test Set', fontsize=13, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison_lsp.png', dpi=150)
plt.show()
print('Figura guardada: model_comparison_lsp.png')

In [ ]:
# ── Curvas de entrenamiento ───────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

models_hist = [('CNN-LSTM', history_cnn), ('ST-GCN', history_st), ('Fusión', history_fusion)]

for col, (name, hist) in enumerate(models_hist):
    # Loss
    axes[0, col].plot(hist.get('train_loss', []), label='Train', lw=2)
    axes[0, col].plot(hist.get('val_loss', []), label='Val', lw=2, ls='--')
    axes[0, col].set_title(f'{name} — Loss')
    axes[0, col].legend()
    axes[0, col].grid(True, alpha=0.3)

    # Accuracy
    axes[1, col].plot(hist.get('train_accuracy', []), label='Train', lw=2, color='steelblue')
    axes[1, col].plot(hist.get('val_accuracy', []), label='Val', lw=2, color='orange', ls='--')
    axes[1, col].set_title(f'{name} — Accuracy')
    axes[1, col].legend()
    axes[1, col].grid(True, alpha=0.3)

plt.suptitle('Curvas de Entrenamiento — Arquitecturas LSP', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves_lsp.png', dpi=150)
plt.show()

In [ ]:
# ── Exportar el mejor modelo a ONNX ──────────────────────────────────
best_model_name = df_results['accuracy'].idxmax()
print(f'Mejor modelo: {best_model_name}')

# Seleccionar trainer del mejor modelo
trainer_map = {'CNN-LSTM': trainer_cnn, 'ST-GCN': trainer_st, 'Fusión': trainer_fusion}
best_trainer = trainer_map[best_model_name]

ckpt_map = {'CNN-LSTM': 'cnn_lstm', 'ST-GCN': 'stgcn', 'Fusión': 'fusion'}
ckpt_name = ckpt_map[best_model_name]

best_trainer.export_onnx(
    checkpoint_path=f'{CKPT_DIR}/{ckpt_name}_best.pt',
    output_path=f'{CKPT_DIR}/lsp_model.onnx',
    opset_version=17,
)
print(f'Modelo ONNX exportado: {CKPT_DIR}/lsp_model.onnx')

In [ ]:
# ── Benchmark de latencia ─────────────────────────────────────────────
import time
import torch.nn.functional as F

N_RUNS = 50
latencies = {}

for model_name, trainer in [('CNN-LSTM', trainer_cnn), ('ST-GCN', trainer_st)]:
    model = trainer.model.eval().to(DEVICE)
    
    if model_name == 'CNN-LSTM':
        dummy = torch.randn(1, 3, 30, 224, 224).to(DEVICE)
        fn = lambda: model(dummy)
    else:
        dummy = torch.randn(1, 30, 75, 3).to(DEVICE)
        fn = lambda: model(dummy)
    
    # Warm-up
    for _ in range(5):
        with torch.no_grad():
            fn()
    
    # Benchmark
    times = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        with torch.no_grad():
            fn()
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    
    latencies[model_name] = {
        'mean_ms': np.mean(times),
        'p95_ms':  np.percentile(times, 95),
        'min_ms':  np.min(times),
    }

print('\nBENCHMARK DE LATENCIA DE INFERENCIA')
print('-' * 40)
for name, lat in latencies.items():
    status = '✓' if lat['mean_ms'] < 200 else '✗'
    print(f'{name:12s} | Media: {lat["mean_ms"]:6.1f}ms | P95: {lat["p95_ms"]:6.1f}ms {status}')
print(f'Objetivo: <200ms para tiempo real')